In [1]:
import os
import json
import glob
import pandas as pd
import re
from pathlib import Path
import xlsxwriter
import openpyxl
from openpyxl import load_workbook

In [2]:
def get_DGCES_to_ML(directory, state_cd, season_cd):
    #df to use
    df_state = pd.read_excel("ML_Template.xlsx", sheet_name="State", dtype=str)
    state = df_state['STATE'].iloc[0]
    year = df_state['YEAR'].iloc[0]
    
    df_samples = pd.read_excel("ML_Template.xlsx", sheet_name="Samples", dtype=str)
    df_districts = pd.read_excel("ML_Template.xlsx", sheet_name="Districts", dtype=str)
    df_crops = pd.read_excel("ML_Template.xlsx", sheet_name="Crops", dtype=str)

    #df to create
    df_Vill = pd.DataFrame(columns=['STATE','SEASONCODE','SAMPLE','CROPNAME',
                                    'DISTRICTNAME','BLOCK','VILLAGE','VILLAGECODE'])

    # Get all .xlsx and .xls files in the directory
    for file_path in directory.iterdir():
        if file_path.suffix == '.xlsx':
            df_raw = pd.read_excel(file_path, sheet_name="District Village Wise Plan", dtype=str)
            df = df_raw[df_raw['Final Selected Villages'].notna()]
            
            if 'Regional Name' in df.columns:
                sample_cd = '1'
            else:
                sample_cd = '2'
            
            for row1 in df.iterrows():
                row = row1[1]
                crop_name = row['Crop Name']
                district_name = row['District Name']
                
                vill_list_str = row['Final Selected Villages']
                vill_list = vill_list_str.split(',')
                for vill in vill_list:
                    vill_name = re.search(r"^(.*?)\s*\(\d+\)\s*$", vill).group(1)
                    vill_code = re.search(r"\((\d+)\)\s*$", vill)

                    numbers = re.findall(r'\((\d+)\)', vill)
                    vill_code = 0
                    for number in numbers:
                        if len(str(number)) >= 5:
                            vill_code = number
                            
                            break
                            
                    df_village_row = {
                        'STATE': str(state_cd),
                        'DISTRICTNAME': district_name,
                        'SAMPLE': str(sample_cd),
                        'SEASONCODE': str(season_cd),
                        'CROPNAME': crop_name,
                        'BLOCK': None,
                        'VILLAGE': vill_name,
                        'VILLAGECODE': str(vill_code)
                    }
                    df_Vill = pd.concat([df_Vill, pd.DataFrame([df_village_row])], ignore_index=True)
    return df_Vill

In [3]:
if __name__ == "__main__":
    state_cd = '33'
    season_cd = '1'
    curr_dir = Path.cwd()
    directory = curr_dir / "DGCES Auto Village Selection"
    excel_path = curr_dir / "DGCES_to_ML.xlsx"
    
    df_Vill = get_DGCES_to_ML(directory, state_cd, season_cd)

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        df_Vill.to_excel(writer, sheet_name="Villages", index=False)